[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/%E6%A9%9F%E7%8E%87%E7%B5%B1%E8%A8%88/08_%E5%88%86%E4%BD%88%E6%AA%A2%E6%9F%A5%E8%88%87p%E5%80%BC.ipynb) · [回章節首頁](./README.md)

## 這個觀念要幫我們回答什麼？

**核心問題：資料不貼常態參考線，是資料壞了，還是參照選錯了？p 值又該怎麼讀才不會誤判？**

- **用在流程哪裡**：分布假設的正式檢查（圖形＋檢定並用），以及檢定結果的正確表述。
- **讀完要能做到**：能區分「不符合某參照」與「資料錯誤」，正確解讀檢定的證據與限制。
- **閱讀方式**：每個程式格前先讀「為什麼做／怎麼看」，再執行；用實際輸出回答「先想一想」。

觀念連結：[特徵工程](../名詞解釋/04_特徵工程/README.md)、[模型泛化](../名詞解釋/11_模型泛化/README.md)

# 08｜分布檢查：先選參照，再解讀 p 值

07 學會看形狀，本篇學會「**換參照**」與「**讀 p 值**」——Q-Q 圖最常見的兩個坑。

**一、同一份資料，參照不同結論不同**

四種合成資料（常態、均勻、指數、對數常態）先**全部對常態參照**作圖：只有常態貼線，其餘彎曲——但彎曲 $\neq$ 資料錯誤，可能只是**參照不合適**（拿直尺量圓）。改用各自正確的參照（norm／uniform／expon／lognorm，後者形狀參數 `s=1`）後，四組都貼線。所以「S 形就是均勻」這種口訣一定要追問：**跟誰比？**圖名是資料來源，橫軸永遠是當次選定的理論分位數。

**二、Shapiro 檢定：沒拒絕 $\neq$ 證明**

- **H0**：樣本來自常態分布；固定 $\alpha = 0.05$ 當裁判線，不挑最好看的 p 值。
- **p 值的正確讀法**：「在 H0 及檢定假設下，看到至少這麼極端統計量的機率」。$p = 0.2$ 絕對不是「有 20% 機率是常態」；結論只能說「未拒絕常態假設，**不是證明**」。
- 本例每組 800 筆，在 SciPy 建議的 5000 筆範圍內；筆數越大，微小偏離越容易顯著——**顯著 $\neq$ 重要**。

**三、KS 檢定：參數要事先說清楚**

KS 比較樣本與「**事先指定的完整 CDF**」（本篇常態指定 $N(0,1)$、均勻 $U(0,1)$ 等，參數來自生成設定）。若用**同一份資料估參數再套一般 KS 的 p 值**，p 值會失真，需用能處理估參數的版本或重抽樣方法。KS 也不保證獨立性，更不會告訴你「唯一正確的分布」。

**四、接回實務**：四組都是合成示例，已知生成分布也可能因隨機波動被拒絕。實務要合看情境、圖形、樣本量與模型驗證；分布是拿來做決定的工具，不是拿來背誦的标签。

## 準備共用繪圖設定

**為什麼做**：統一繁體中文字型與配色，讓讀圖重點清楚。

**怎麼看**：這格只載入工具，沒有訓練模型；會列出實際字型。需要套件時由老師協助準備 NumPy、pandas、SciPy、Matplotlib、Seaborn。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

# 設定中文字型支援
# --- 設定 Matplotlib 中文字型支援（Mac / Windows / Colab 通用）---
import glob
import os
try:
    import mplfonts
except ImportError:  # Colab 環境若缺套件則自動安裝（本機已在 requirements.txt）
    %pip install -q mplfonts
    import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)   # 直接註冊 mplfonts 內附的字型，不改任何檔案
use_font("Noto Sans CJK SC")              # 套用思源黑體，負號也會一起處理

sns.set_theme(style='whitegrid', rc={
    'axes.unicode_minus': False, 'figure.facecolor': '#F5F7F2',
    'axes.facecolor': '#FFFFFF', 'axes.spines.top': False,
    'axes.spines.right': False, 'grid.alpha': 0.22,
    'axes.titlesize': 14, 'axes.labelsize': 11, 'figure.dpi': 110})
GREEN, ORANGE, PURPLE = '#2F6657', '#C17B45', '#786497'
rng = np.random.default_rng(42)
print('中文字型：Noto Sans CJK SC（mplfonts 通用寫法）')

## 01｜同一張常態參考圖，不適合所有形狀

**為什麼做**：產生四種已知分布，全部先與常態參照比較，觀察參照選擇的重要性。

**怎麼看**：圖名是資料來源，不是參照；四圖的橫軸都仍是常態理論分位數。彎曲不等於資料錯誤，可能只是參照不合適。

In [ ]:
rng = np.random.default_rng(42)
datasets = {'常態': rng.normal(size=800), '均勻': rng.uniform(0, 1, 800),
            '指數': rng.exponential(scale=1, size=800),
            '對數常態': rng.lognormal(mean=0, sigma=1, size=800)}
fig, axes = plt.subplots(2, 2, figsize=(10, 7), layout='constrained')
for ax, (name, values) in zip(axes.flat, datasets.items()):
    (theoretical, ordered), (slope, intercept, r) = stats.probplot(values, dist='norm')
    ax.scatter(theoretical, ordered, s=9, color=GREEN, alpha=.35)
    ax.plot(theoretical, slope * theoretical + intercept, color=ORANGE)
    ax.set(title=f'{name}資料／常態參照', xlabel='常態理論分位數', ylabel='樣本分位數')
plt.show()

## 02｜改用各自的分布參照

**為什麼做**：改參照，而不改樣本，確認同一份資料的判讀依賴比較問題。

**怎麼看**：這次四種資料分別對 norm、uniform、expon、lognorm；對數常態指定形狀參數 s=1。不能只背「S 形就是均勻」而不問比較對象。

In [ ]:
references = [('norm', ()), ('uniform', ()), ('expon', ()), ('lognorm', (1,))]
fig, axes = plt.subplots(2, 2, figsize=(10, 7), layout='constrained')
for ax, (name, values), (distribution, params) in zip(axes.flat, datasets.items(), references):
    (theoretical, ordered), (slope, intercept, r) = stats.probplot(values, dist=distribution, sparams=params)
    ax.scatter(theoretical, ordered, s=9, color=GREEN, alpha=.35)
    ax.plot(theoretical, slope * theoretical + intercept, color=ORANGE)
    ax.set(title=f'{name}資料／相同分布參照', xlabel='該分布理論分位數', ylabel='樣本分位數')
plt.show()

## 03｜Shapiro：沒有拒絕不等於證明

**為什麼做**：檢定零假設：樣本來自常態分布。以 α=0.05 示範固定標準，不用「挑最好看的 p 值」選模型。

**怎麼看**：p 值是在零假設及檢定假設下，得到至少如此極端統計量的機率，不是「資料常態的機率」。本例每組800筆，不超過 SciPy 提醒的 5,000 筆範圍。

In [ ]:
alpha = .05
rows = []
for name in ['常態', '對數常態']:
    statistic, p_value = stats.shapiro(datasets[name])
    decision = '拒絕常態假設' if p_value <= alpha else '未拒絕常態假設；不是證明'
    rows.append({'樣本': name, '統計量': statistic, 'p 值': p_value, '在 α=0.05 的判讀': decision})
display(pd.DataFrame(rows))

## 04｜KS：理論參數要事先說清楚

**為什麼做**：比較樣本與事先指定的完整 CDF。本例的參數是生成時已知，沒有用同一份樣本估計後直接套一般 KS p 值。

**怎麼看**：常態指定 N(0,1)、均勻指定 U(0,1)、指數尺度為1、對數常態 s=1。KS 不會替你保證獨立性，也不會找出唯一正確分布。

In [ ]:
ks_specs = [('常態', 'norm', (0, 1)), ('均勻', 'uniform', (0, 1)),
            ('指數', 'expon', (0, 1)), ('對數常態', 'lognorm', (1, 0, 1))]
rows = []
for name, distribution, params in ks_specs:
    result = stats.kstest(datasets[name], distribution, args=params)
    decision = '拒絕指定分布' if result.pvalue <= .05 else '未拒絕指定分布；不是證明'
    rows.append({'資料／事先指定參照': name, 'KS 統計量': result.statistic,
                 'p 值': result.pvalue, '判讀': decision})
display(pd.DataFrame(rows))

## 接回機器學習

四組資料都是合成示例，已知生成分布也可能因隨機波動被拒絕。實務要合看情境、圖形、樣本量與模型驗證；用資料估參數時，需採用能處理估參數的檢定或重抽樣方法。

## 換你解讀

1. 均勻資料不貼常態線，是資料壞了還是參照不合？
2. p=0.2 可以說有 20% 機率常態嗎？
3. 為什麼不能把同一份資料估出的參數，直接套一般 KS 的標準判讀？

### 請 AI 幫你做一個小實驗

請 AI 對同一份右偏資料畫常態與對數常態參照圖，保持資料不變。說明你改變的是資料、模型，還是比較問題。

先說出預期，再看實際圖形，最後寫下「證據 → 解釋 → 不能下的結論」。

[回章節首頁](./README.md)